# Fase 09 — Migracao: api-llm-embedded -> monorepo-ai-llm

**Data:** 2026-05-02  
**Contexto:** Migracao completa do projeto `api-llm-embedded` (npm workspaces, NestJS monolito multi-dominio, 5 MCP servers) para o monorepo `monorepo-ai-llm` (NestJS CLI monorepo, 5 apps separados).  
**Documento de referencia:** `spec/fases/fase-09-plano-migracao-api-llm-embedded.md`
**Padrao de raiz:** este projeto deve ser executado dentro de `/mnt/repositorio/workdir/repostorios/<repositorio>`.

---

## Indice

1. [Contexto e Diagnostico](#contexto)
2. [Fase 1 — Fundacao: npm Workspaces + Shared Packages](#fase1)
3. [Fase 2 — Infraestrutura de Banco de Dados](#fase2)
4. [Fase 3 — Migracao: users-api](#fase3)
5. [Fase 4 — Migracao: llm-ops-api](#fase4)
6. [Fase 5 — Migracao: sharepoint-api](#fase5)
7. [Fase 6 — Migracao: sync-api](#fase6)
8. [Fase 7 — Migracao: MCP Servers](#fase7)
9. [Fase 8 — Scripts, Config e Docs](#fase8)
10. [Fase 9 — CLAUDE.md](#fase9)
11. [Fase 10 — Validacao Final](#fase10)

## 1. Contexto e Diagnóstico <a id='contexto'></a>

### Projetos

| | Origem | Destino |
|---|---|---|
| **Caminho** | `/mnt/repositorio/workdir/repostorios/api-llm-embedded` | `/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm` |
| **Build** | npm workspaces + `tsgo` | NestJS CLI + webpack |
| **Apps** | `apps/api` monolito + `apps/svcia/` | 5 NestJS apps separados |
| **DB** | TypeORM + migrations | Sem banco ainda |
| **Shared** | `packages/shared` | Não existe |
| **MCP** | 5 servers em `apps/svcia/` | Stub em `tools/mcp/` |

In [ ]:
# Diagnóstico inicial — executar antes de começar
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

echo "=== Estrutura da ORIGEM ==="
ls "$ORIGEM"

echo ""
echo "=== Estrutura do DESTINO ==="
ls "$DESTINO"

echo ""
echo "=== Apps no DESTINO ==="
ls "$DESTINO/apps"

echo ""
echo "=== Node.js e npm ==="
node --version
npm --version

: 

## 2. Fase 1 — Fundação: npm Workspaces + Shared Packages <a id='fase1'></a>

**Objetivo:** Configurar a base de compartilhamento de código entre os apps do monorepo.

**Artefatos a criar:**
- `packages/shared/` — Contratos e tipos compartilhados
- `packages/infra/` — Guards, interceptors, decorators compartilhados
- npm workspaces configurados no `package.json` raiz

In [ ]:
# Fase 1.1 — Verificar se npm workspaces já estão configurados
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

echo "=== package.json raiz (campo workspaces) ==="
node -e "const p = require('./package.json'); console.log('workspaces:', JSON.stringify(p.workspaces, null, 2))"

In [ ]:
# Fase 1.2 — Criar estrutura packages/shared (copiar da origem)
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

# Criar diretório packages/
mkdir -p "$DESTINO/packages"

# Copiar packages/shared da origem
cp -r "$ORIGEM/packages/shared" "$DESTINO/packages/shared"

echo "=== Estrutura do packages/shared criado ==="
find "$DESTINO/packages/shared" -type f | head -30

In [ ]:
# Fase 1.3 — Renomear pacote de @api-llm-embedded/shared → @monorepo-ai-llm/shared
SHARED_PKG="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm/packages/shared/package.json"

# Ver package.json atual
cat "$SHARED_PKG"

# Substituir nome do pacote
# ATENÇÃO: execute manualmente via editor para garantir JSON válido
echo ""
echo "AÇÃO MANUAL: editar $SHARED_PKG e trocar o campo name para @monorepo-ai-llm/shared"

In [ ]:
# Fase 1.4 — Criar packages/infra (extrair de apps/api/src/common/)
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

mkdir -p "$DESTINO/packages/infra/src"

# Copiar decorators, guards, filters, interceptors
for dir in decorators guards filters interceptors; do
  cp -r "$ORIGEM/apps/api/src/common/$dir" "$DESTINO/packages/infra/src/$dir"
  echo "Copiado: $dir"
done

echo ""
echo "=== Estrutura packages/infra ==="
find "$DESTINO/packages/infra" -type f

In [ ]:
# Fase 1.5 — Verificar npm install após configurar workspaces
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

# Instalar dependências do workspace
npm install

echo "Código de saída: $?"

In [ ]:
# Fase 1 — Validação: build ainda funciona após configurar workspaces
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

npm run build:users-api
echo "users-api build: $?"

npm run build:llm-ops-api
echo "llm-ops-api build: $?"

## 3. Fase 2 — Infraestrutura de Banco de Dados <a id='fase2'></a>

**Objetivo:** Configurar TypeORM, PostgreSQL e variáveis de ambiente.

**Artefatos a criar:**
- `docker-compose.yml` com postgres + langflow
- `.env.example` no root
- Módulo database em cada app que precisa de DB

In [ ]:
# Fase 2.1 — Instalar dependências de banco de dados
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

npm install @nestjs/typeorm typeorm pg @nestjs/config class-validator class-transformer zod
npm install --save-dev @types/pg

echo "Dependências instaladas: $?"

In [ ]:
# Fase 2.2 — Verificar docker-compose.yml criado
cat /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm/docker-compose.yml

In [ ]:
# Fase 2.3 — Subir postgres para testes locais
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

docker compose up -d postgres
echo "PostgreSQL status: $?"

# Aguardar postgres ficar pronto
sleep 3
docker compose ps

In [ ]:
# Fase 2.4 — Copiar módulo database da origem para users-api (exemplo)
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

mkdir -p "$DESTINO/apps/users-api/src/infra/database"

# Copiar módulo de database
cp "$ORIGEM/apps/api/src/infra/database/database.module.ts" \
   "$DESTINO/apps/users-api/src/infra/database/"

cp "$ORIGEM/apps/api/src/infra/database/typeorm.config.ts" \
   "$DESTINO/apps/users-api/src/infra/database/"

echo "Copiados. ATENÇÃO: editar typeorm.config.ts para usar as envs corretas."

# Listar migrations de users da origem
echo ""
echo "=== Migrations relacionadas a Users na origem ==="
ls "$ORIGEM/apps/api/src/infra/database/migrations/" | grep -i -E '(user|api.key|permission|govern)' || echo "(nenhuma encontrada com esses filtros — verificar manualmente)"

## 4. Fase 3 — Migração: users-api <a id='fase3'></a>

**Objetivo:** Migrar módulos `users`, `admin`, `governance`, `api-keys` para `apps/users-api`.

**Schema PostgreSQL:** `public`  
**Porta:** 3001

In [ ]:
# Fase 3.1 — Listar módulos a migrar na origem
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"

for mod in users admin governance api-keys health; do
  echo "=== Módulo: $mod ==="
  find "$ORIGEM/apps/api/src/modules/$mod" -type f -name '*.ts' | head -10
  echo ""
done

In [ ]:
# Fase 3.2 — Copiar módulos para users-api
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

mkdir -p "$DESTINO/apps/users-api/src/modules"

for mod in users admin governance api-keys health; do
  if [ -d "$ORIGEM/apps/api/src/modules/$mod" ]; then
    cp -r "$ORIGEM/apps/api/src/modules/$mod" "$DESTINO/apps/users-api/src/modules/$mod"
    echo "✓ Copiado: $mod"
  else
    echo "✗ Não encontrado: $mod"
  fi
done

In [ ]:
# Fase 3.3 — Identificar imports de @api-llm-embedded/shared que precisam ser atualizados
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

echo "=== Arquivos com @api-llm-embedded/shared em users-api ==="
grep -rl '@api-llm-embedded/shared' "$DESTINO/apps/users-api/src/" 2>/dev/null | head -20

echo ""
echo "AÇÃO: substituir @api-llm-embedded/shared por @monorepo-ai-llm/shared nos arquivos acima"
echo "Comando: find ... -exec sed -i 's/@api-llm-embedded\/shared/@monorepo-ai-llm\/shared/g' {} +"

In [ ]:
# Fase 3.4 — Tentar build de users-api após migração
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

npm run build:users-api 2>&1 | tail -30
echo "Build exit code: $?"

In [ ]:
# Fase 3.5 — Smoke test users-api (requer PostgreSQL rodando)
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

# Iniciar app em background e testar
nest start users-api &
APP_PID=$!

sleep 5

# Teste básico de health
curl -s http://localhost:3001/health
echo ""

# Encerrar app
kill $APP_PID 2>/dev/null
echo "Smoke test concluído"

## 5. Fase 4 — Migração: llm-ops-api <a id='fase4'></a>

**Objetivo:** Migrar módulo `llm-ops` (10 entities, RAG, AstraDB, LangFlow) e `audit` para `apps/llm-ops-api`.

**Schema PostgreSQL:** `llm_ops`  
**Porta:** 3002

**Integração RAG:**
```
llm-ops-api → AstraDB (knowledge_base/interactions) → LangFlow (context-only)
Flow ID: f81c0124-ffc2-4458-b30d-4d588d393518
```

In [ ]:
# Fase 4.1 — Listar entidades do domínio llm-ops na origem
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"

echo "=== Entidades em llm-ops ==="
find "$ORIGEM/apps/api/src/modules/llm-ops" -name '*.entity.ts' -type f

echo ""
echo "=== DTOs em llm-ops ==="
find "$ORIGEM/apps/api/src/modules/llm-ops" -name '*.dto.ts' -type f | wc -l
echo "DTOs encontrados"

echo ""
echo "=== AstraDB client ==="
ls "$ORIGEM/apps/api/src/infra/astradb/" 2>/dev/null || echo "(verificar caminho)"

In [ ]:
# Fase 4.2 — Copiar módulos para llm-ops-api
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

mkdir -p "$DESTINO/apps/llm-ops-api/src/modules"
mkdir -p "$DESTINO/apps/llm-ops-api/src/infra"

# Módulos de negócio
for mod in llm-ops audit health; do
  if [ -d "$ORIGEM/apps/api/src/modules/$mod" ]; then
    cp -r "$ORIGEM/apps/api/src/modules/$mod" "$DESTINO/apps/llm-ops-api/src/modules/$mod"
    echo "✓ Copiado módulo: $mod"
  fi
done

# Infra: AstraDB
if [ -d "$ORIGEM/apps/api/src/infra/astradb" ]; then
  cp -r "$ORIGEM/apps/api/src/infra/astradb" "$DESTINO/apps/llm-ops-api/src/infra/astradb"
  echo "✓ Copiado: infra/astradb"
fi

# RAG LangFlow
LANGFLOW_SRC="$ORIGEM/apps/svcia/mcp-llm-ops/src/rag-langflow"
if [ -d "$LANGFLOW_SRC" ]; then
  mkdir -p "$DESTINO/apps/llm-ops-api/src/rag-langflow"
  cp -r "$LANGFLOW_SRC/" "$DESTINO/apps/llm-ops-api/src/rag-langflow/"
  echo "✓ Copiado: rag-langflow"
fi

In [ ]:
# Fase 4.3 — Build de llm-ops-api
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

npm run build:llm-ops-api 2>&1 | tail -30
echo "Build exit code: $?"

## 6. Fase 5 — Migração: sharepoint-api <a id='fase5'></a>

**Objetivo:** Migrar `sharepoint` e `graph` (Microsoft Graph) para `apps/sharepoint-api`.

**App stateless** — sem banco de dados.  
**Porta:** 3003  
**Auth:** Microsoft Entra ID + OAuth + certificado

In [ ]:
# Fase 5.1 — Copiar módulos para sharepoint-api
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

mkdir -p "$DESTINO/apps/sharepoint-api/src/modules"

for mod in sharepoint graph health; do
  if [ -d "$ORIGEM/apps/api/src/modules/$mod" ]; then
    cp -r "$ORIGEM/apps/api/src/modules/$mod" "$DESTINO/apps/sharepoint-api/src/modules/$mod"
    echo "✓ Copiado: $mod"
  fi
done

echo ""
echo "ATENÇÃO: sharepoint-api é STATELESS — não incluir DatabaseModule"

In [ ]:
# Fase 5.2 — Build de sharepoint-api
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

npm run build:sharepoint-api 2>&1 | tail -30
echo "Build exit code: $?"

## 7. Fase 6 — Migração: sync-api <a id='fase6'></a>

**Objetivo:** Migrar `sync` e `m365-migration` (8 entities) para `apps/sync-api`.

**Schema PostgreSQL:** `public`  
**Porta:** 3004

In [ ]:
# Fase 6.1 — Copiar módulos para sync-api
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

mkdir -p "$DESTINO/apps/sync-api/src/modules"

for mod in sync m365-migration health; do
  if [ -d "$ORIGEM/apps/api/src/modules/$mod" ]; then
    cp -r "$ORIGEM/apps/api/src/modules/$mod" "$DESTINO/apps/sync-api/src/modules/$mod"
    echo "✓ Copiado: $mod"
  fi
done

In [ ]:
# Fase 6.2 — Build de sync-api
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

npm run build:sync-api 2>&1 | tail -30
echo "Build exit code: $?"

## 8. Fase 7 — Migração: MCP Servers <a id='fase7'></a>

**Objetivo:** Migrar os 5 servidores MCP de `apps/svcia/` e o `tools/mcp-project-health`.

**Todos os MCP servers são read-only** — nenhuma mutação via MCP.

| Server | Origem | Destino |
|---|---|---|
| mcp-users | `apps/svcia/mcp-users/` | `apps/svcia/mcp-users/` |
| mcp-llm-ops | `apps/svcia/mcp-llm-ops/` | `apps/svcia/mcp-llm-ops/` |
| mcp-sync | `apps/svcia/mcp-sync/` | `apps/svcia/mcp-sync/` |
| mcp-secrets | `apps/svcia/mcp-secrets/` | `apps/svcia/mcp-secrets/` |
| mcp-awesome-copilot | `apps/mcp-awesome-copilot/` | `apps/svcia/mcp-awesome-copilot/` |
| mcp-project-health | `tools/mcp-project-health/` | `tools/mcp-project-health/` |

In [ ]:
# Fase 7.1 — Copiar apps/svcia/ para o monorepo
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

mkdir -p "$DESTINO/apps/svcia"

# Copiar cada MCP server
for server in mcp-users mcp-llm-ops mcp-sync mcp-secrets; do
  if [ -d "$ORIGEM/apps/svcia/$server" ]; then
    cp -r "$ORIGEM/apps/svcia/$server" "$DESTINO/apps/svcia/$server"
    echo "✓ Copiado: $server"
  fi
done

# mcp-awesome-copilot tem localização diferente na origem
if [ -d "$ORIGEM/apps/mcp-awesome-copilot" ]; then
  cp -r "$ORIGEM/apps/mcp-awesome-copilot" "$DESTINO/apps/svcia/mcp-awesome-copilot"
  echo "✓ Copiado: mcp-awesome-copilot"
fi

echo ""
echo "=== Estrutura apps/svcia no destino ==="
ls "$DESTINO/apps/svcia/"

In [ ]:
# Fase 7.2 — Copiar tools/mcp-project-health
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

cp -r "$ORIGEM/tools/mcp-project-health" "$DESTINO/tools/mcp-project-health"
echo "✓ Copiado: tools/mcp-project-health"

echo ""
echo "=== Estrutura tools/ no destino ==="
ls "$DESTINO/tools/"

In [ ]:
# Fase 7.3 — Renomear packages em todos os MCP servers
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

echo "=== package.json dos MCP servers (campo name) ==="
find "$DESTINO/apps/svcia" -name 'package.json' -maxdepth 3 | while read f; do
  NAME=$(node -e "console.log(require('$f').name)" 2>/dev/null)
  echo "$f: $NAME"
done

echo ""
echo "AÇÃO: substituir @api-llm-embedded/ por @monorepo-ai-llm/ em cada package.json"

In [ ]:
# Fase 7.4 — Build dos MCP servers (cada um usa seu próprio build script)
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

for server in mcp-users mcp-llm-ops mcp-sync mcp-secrets; do
  echo "=== Build: $server ==="
  cd "$DESTINO/apps/svcia/$server"
  npm run build 2>&1 | tail -5
  echo "Exit code: $?"
  echo ""
done

In [ ]:
# Fase 7.5 — Smoke test dos MCP servers (requer apis rodando)
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

# Testar mcp-users (requer users-api em :3001)
if [ -f "$DESTINO/scripts/smoke-tests/mcp-users-read-smoke.mjs" ]; then
  node "$DESTINO/scripts/smoke-tests/mcp-users-read-smoke.mjs"
else
  echo "Smoke test ainda não migrado para o destino — executar Fase 8 primeiro"
fi

## 9. Fase 8 — Scripts, Config e Docs <a id='fase8'></a>

**Objetivo:** Migrar scripts de smoke tests, configurações MCP e documentação de arquitetura.

In [ ]:
# Fase 8.1 — Copiar scripts/
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

mkdir -p "$DESTINO/scripts"

# Copiar smoke tests
cp -r "$ORIGEM/scripts/smoke-tests" "$DESTINO/scripts/smoke-tests"
echo "✓ Copiado: scripts/smoke-tests"

# Copiar bootstrap scripts
cp -r "$ORIGEM/scripts/bootstrap" "$DESTINO/scripts/bootstrap" 2>/dev/null && echo "✓ Copiado: scripts/bootstrap"

echo ""
echo "AÇÃO: atualizar caminhos hardcoded que referenciam 'api-llm-embedded' nos .mjs"
grep -r 'api-llm-embedded' "$DESTINO/scripts/" 2>/dev/null | head -10

In [ ]:
# Fase 8.2 — Copiar config/mcp (templates de configuração)
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

mkdir -p "$DESTINO/config"
cp -r "$ORIGEM/config/mcp" "$DESTINO/config/mcp"
echo "✓ Copiado: config/mcp"

echo ""
echo "=== Config MCP copiados ==="
ls "$DESTINO/config/mcp/"

In [ ]:
# Fase 8.3 — Copiar docs de arquitetura
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

if [ -d "$ORIGEM/docs" ]; then
  mkdir -p "$DESTINO/docs"
  cp -r "$ORIGEM/docs/" "$DESTINO/docs/"
  echo "✓ Copiado: docs/"
  ls "$DESTINO/docs/"
fi

## 10. Fase 9 — CLAUDE.md <a id='fase9'></a>

**Objetivo:** Criar CLAUDE.md no root do monorepo baseado no projeto origem, adaptado para a nova estrutura.

O `CLAUDE.md` é lido automaticamente pelo Claude Code em cada sessão. Ele deve refletir a arquitetura **atual** do monorepo após a migração.

In [ ]:
# Fase 9.1 — Verificar se CLAUDE.md existe no destino
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

if [ -f "$DESTINO/CLAUDE.md" ]; then
  echo "CLAUDE.md já existe. Tamanho:"
  wc -l "$DESTINO/CLAUDE.md"
else
  echo "CLAUDE.md NÃO existe — precisa ser criado"
  echo "Referência: /mnt/repositorio/workdir/repostorios/api-llm-embedded/CLAUDE.md"
fi

In [ ]:
# Fase 9.2 — Criar CLAUDE.md baseado na origem
# AÇÃO MANUAL: criar ou editar CLAUDE.md no destino
# Pontos-chave a adaptar:

echo "Checklist CLAUDE.md:"
echo "[ ] Atualizar nomes de pacotes: @api-llm-embedded/ → @monorepo-ai-llm/"
echo "[ ] Atualizar estrutura de apps (5 NestJS apps separados)"
echo "[ ] Atualizar comandos de build (NestJS CLI em vez de tsgo direto)"
echo "[ ] Atualizar caminhos de MCP servers (apps/svcia/)"
echo "[ ] Remover referências a tsgo como compilador principal"
echo "[ ] Manter seções de regras de governance e segurança"

## 11. Fase 10 — Validação Final <a id='fase10'></a>

**Objetivo:** Garantir que o monorepo migrado está funcional em todos os aspectos.

In [ ]:
# Fase 10.1 — Build completo
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

echo "=== Build de todos os apps ==="
for app in users-api llm-ops-api sharepoint-api sync-api; do
  echo "Building $app..."
  npm run build:$app 2>&1 | tail -3
  echo "Exit: $?"
  echo ""
done

In [ ]:
# Fase 10.2 — Type check
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

npx tsc --noEmit 2>&1 | head -50
echo "Type check exit code: $?"

In [ ]:
# Fase 10.3 — Lint
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

npm run lint 2>&1 | tail -20
echo "Lint exit code: $?"

In [ ]:
# Fase 10.4 — Smoke tests (requer PostgreSQL rodando)
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

# Verificar se postgres está rodando
docker compose ps postgres

# Rodar smoke tests se disponíveis
if [ -f "scripts/smoke-tests/run-all.mjs" ]; then
  node scripts/smoke-tests/run-all.mjs
else
  echo "Smoke test runner não migrado ainda — executar individualmente"
  ls scripts/smoke-tests/ | head -10
fi

In [ ]:
# Fase 10.5 — Docker build validation
cd /mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm

docker compose -f docker-compose.remote.yml build --no-cache 2>&1 | tail -30
echo "Docker build exit code: $?"

In [ ]:
# Fase 10.6 — Checklist final de validacao
echo "=== CHECKLIST FINAL DE MIGRACAO ==="
echo ""

DESTINO="/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm"

# Verificar artefatos obrigatorios
check_exists() {
  if [ -e "$DESTINO/$1" ]; then
    echo "[OK] $1"
  else
    echo "[FALTA] $1"
  fi
}

check_exists "packages/shared/src/index.ts"
check_exists "packages/infra/src/index.ts"
check_exists "apps/users-api/src/modules/users"
check_exists "apps/llm-ops-api/src/modules/llm-ops"
check_exists "apps/sharepoint-api/src/modules/sharepoint"
check_exists "apps/sync-api/src/modules/sync"
check_exists "apps/svcia/mcp-users"
check_exists "apps/svcia/mcp-llm-ops"
check_exists "apps/svcia/mcp-sync"
check_exists "apps/svcia/mcp-secrets"
check_exists "tools/mcp-project-health"
check_exists "scripts/smoke-tests"
check_exists "config/mcp"
check_exists "docker-compose.yml"
check_exists ".env.example"
check_exists "CLAUDE.md"
check_exists "spec/fases/fase-09-plano-migracao-api-llm-embedded.md"
check_exists "spec/execucao/fase-09-migracao-api-llm-embedded.ipynb"

## Conclusao

A migracao completa do `api-llm-embedded` para o `monorepo-ai-llm` e concluida quando todos os itens do checklist final estao marcados e os builds passam sem erros.

### Proximos passos apos a migracao

1. Atualizar configuracoes de MCP no VS Code para apontar para os novos caminhos
2. Configurar GitHub Actions para CI dos novos apps
3. Criar PR documentado com evidencias de validacao
4. Arquivar ou deprecar o repositorio `api-llm-embedded`

### Artefatos desta migracao

- `spec/fases/fase-09-plano-migracao-api-llm-embedded.md` — Plano detalhado
- `spec/execucao/fase-09-migracao-api-llm-embedded.ipynb` — Este notebook

---

*Conforme regra obrigatoria do README: toda migracao deve incluir um `.md` e um `.ipynb`.*